## knn-v2.ipynb
Builds an extended KNN recommendation model (v2) by adding two new feature
blocks — artist country and album track statistics — to the baseline feature
set from `knn.ipynb`. Everything else (universe expansion, safe column pruning,
L2 normalisation, brute-force cosine search) follows the original workflow.

New feature sources:
- `album_country_matrix.npz`     (from `weights-country.ipynb`)
- `album_track_stats_matrix.npz` (from `weights-track-stats.ipynb`)

In [2]:
import os
import pickle
import numpy as np
import pandas as pd
import joblib
from scipy.sparse import csr_matrix, hstack, load_npz, save_npz
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize

os.makedirs('../data/features', exist_ok=True)
os.makedirs('../data/model_v2', exist_ok=True)

In [3]:
# Load row index
with open('../data/features/album_ids.pkl', 'rb') as f:
    album_id_order = pickle.load(f)

# Load baseline feature blocks (v1)
X_tags        = load_npz('../data/features/album_tags_matrix.npz')
X_labels      = load_npz('../data/features/album_labels_matrix.npz')
X_types       = load_npz('../data/features/album_types_matrix.npz')
X_ratings     = load_npz('../data/features/album_ratings_matrix.npz')

# Load new feature blocks (v2 additions)
X_country     = load_npz('../data/features/album_country_matrix.npz')
X_track_stats = load_npz('../data/features/album_track_stats_matrix.npz')

print('Feature blocks loaded:')
for name, X in [('X_tags', X_tags), ('X_labels', X_labels), ('X_types', X_types),
                ('X_ratings', X_ratings), ('X_country', X_country), ('X_track_stats', X_track_stats)]:
    print(f'  {name:<18} {str(X.shape):<25} nnz={X.nnz:,}')

Feature blocks loaded:
  X_tags             (1008102, 3041)           nnz=3,004,997
  X_labels           (1008102, 3469)           nnz=402,047
  X_types            (1008102, 10)             nnz=402,047
  X_ratings          (1008102, 1)              nnz=44,334
  X_country          (1008102, 2263)           nnz=883,503
  X_track_stats      (1008102, 12)             nnz=11,139,577


In [4]:
# Expand all matrices to the full album universe (same logic as knn.ipynb)
full_album_ids = pd.Index(
    pd.read_parquet('../data/mb_album.parquet', columns=['id'])['id'].sort_values()
)

all_blocks = {
    'X_tags':        X_tags,
    'X_labels':      X_labels,
    'X_types':       X_types,
    'X_ratings':     X_ratings,
    'X_country':     X_country,
    'X_track_stats': X_track_stats,
}

if len(album_id_order) < len(full_album_ids):
    print(f'Expanding {len(album_id_order):,} → {len(full_album_ids):,} albums...')
    current_pos = full_album_ids.get_indexer(album_id_order)
    n_full = len(full_album_ids)

    def _expand(X, row_pos, n):
        coo = X.tocoo()
        return csr_matrix((coo.data, (row_pos[coo.row], coo.col)), shape=(n, X.shape[1]))

    all_blocks = {k: _expand(v, current_pos, n_full) for k, v in all_blocks.items()}
    album_id_order = full_album_ids.tolist()

X_tags, X_labels, X_types, X_ratings, X_country, X_track_stats = all_blocks.values()

Expanding 1,008,102 → 2,241,402 albums...


In [5]:
# --- Block weights ---------------------------------------------------
# Each scalar scales the entire block before stacking. After L2
# normalisation the absolute values don't matter — only the ratios do.
# A weight of 1.0 is neutral. Lower values reduce a block's pull on
# cosine similarity; higher values increase it.
#
# Country was found to dominate results at weight 1.0 because it is a
# single dense binary column competing against thousands of sparse tag
# columns. Start at 0.2 and tune by comparing v1 vs v2 results in the app.
W_TAGS         = 1.0
W_LABELS       = 1.0
W_TYPES        = 1.0
W_RATINGS      = 1.0
W_COUNTRY      = 0.2   # <-- tune this (try 0.1 – 0.5)
W_TRACK_STATS  = 1.0

# Assemble extended feature matrix with per-block scaling
X_final_v2 = hstack([
    X_tags         * W_TAGS,
    X_labels       * W_LABELS,
    X_types        * W_TYPES,
    X_ratings      * W_RATINGS,
    X_country      * W_COUNTRY,
    X_track_stats  * W_TRACK_STATS,
]).tocsr()

print(f'X_final_v2: {X_final_v2.shape[0]:,} albums x {X_final_v2.shape[1]:,} features  (nnz={X_final_v2.nnz:,})')
print(f'\nFeature block summary (with weights):')
print(f'  tags         w={W_TAGS}   : {X_tags.shape[1]:,} cols')
print(f'  labels       w={W_LABELS}   : {X_labels.shape[1]:,} cols')
print(f'  types        w={W_TYPES}   : {X_types.shape[1]:,} cols')
print(f'  ratings      w={W_RATINGS}   : {X_ratings.shape[1]:,} cols')
print(f'  country      w={W_COUNTRY} : {X_country.shape[1]:,} cols')
print(f'  track_stats  w={W_TRACK_STATS}   : {X_track_stats.shape[1]:,} cols')
print(f'  Total                   : {X_final_v2.shape[1]:,} cols')

X_final_v2: 2,241,402 albums x 8,796 features  (nnz=15,876,505)

Feature block summary (with weights):
  tags         w=1.0   : 3,041 cols
  labels       w=1.0   : 3,469 cols
  types        w=1.0   : 10 cols
  ratings      w=1.0   : 1 cols
  country      w=0.2 : 2,263 cols
  track_stats  w=1.0   : 12 cols
  Total                   : 8,796 cols


In [6]:
# Safe column pruning — identical strategy to knn.ipynb
col_nnz      = np.diff(X_final_v2.tocsc().indptr)
row_lengths  = np.diff(X_final_v2.indptr)
has_features = row_lengths > 0

col_nnz_vals           = col_nnz[X_final_v2.indices]
nonempty_rows          = np.where(has_features)[0]
row_starts             = X_final_v2.indptr[nonempty_rows]
max_col_nnz_per_album  = np.zeros(X_final_v2.shape[0], dtype=col_nnz.dtype)
max_col_nnz_per_album[nonempty_rows] = np.maximum.reduceat(col_nnz_vals, row_starts)

safe_threshold = int(max_col_nnz_per_album[has_features].min())
keep_cols      = col_nnz >= safe_threshold
X_knn_v2       = X_final_v2[:, keep_cols]

print(f'Safe threshold : {safe_threshold}')
print(f'Columns before : {X_final_v2.shape[1]:,}')
print(f'Columns after  : {X_knn_v2.shape[1]:,}  ({keep_cols.mean()*100:.1f}% retained)')
print(f'Albums with features: {has_features.sum():,}  ({has_features.mean()*100:.1f}%)')

Safe threshold : 10
Columns before : 8,796
Columns after  : 5,854  (66.6% retained)
Albums with features: 1,008,102  (45.0%)


In [7]:
# Subset to annotated albums and L2-normalise
X_knn_annotated_v2    = X_knn_v2[has_features].copy()
album_ids_annotated_v2 = np.array(album_id_order)[has_features]

nan_count = np.isnan(X_knn_annotated_v2.data).sum()
if nan_count:
    print(f'Removing {nan_count:,} NaN entries...')
    np.nan_to_num(X_knn_annotated_v2.data, nan=0.0, copy=False)
    X_knn_annotated_v2.eliminate_zeros()

X_knn_norm_v2 = normalize(X_knn_annotated_v2, norm='l2')
print(f'Fitting on {X_knn_norm_v2.shape[0]:,} albums x {X_knn_norm_v2.shape[1]:,} features')

Removing 7,657 NaN entries...
Fitting on 1,008,102 albums x 5,854 features


In [8]:
model_v2 = NearestNeighbors(metric='cosine', algorithm='brute', n_jobs=-1)
model_v2.fit(X_knn_norm_v2)
print('Model v2 fitted.')

Model v2 fitted.


In [9]:
# Sanity check — query the first annotated album
distances, indices = model_v2.kneighbors(X_knn_norm_v2[0], n_neighbors=11)

print(f'Query album id: {album_ids_annotated_v2[0]}')
print(f"\n{'rank':<6} {'album_id':<40} {'cosine distance':>15}")
print('-' * 62)
for rank, (idx, dist) in enumerate(zip(indices[0], distances[0])):
    label = '(query)' if rank == 0 else ''
    print(f"{rank:<6} {str(album_ids_annotated_v2[idx]):<40} {dist:>15.4f}  {label}")

Query album id: 4

rank   album_id                                 cosine distance
--------------------------------------------------------------
0      4                                                 0.0000  (query)
1      576                                               0.0271  
2      1795                                              0.0701  
3      5391                                              0.0881  
4      2229                                              0.0891  
5      25343                                             0.0927  
6      2609                                              0.0942  
7      1704                                              0.0950  
8      131486                                            0.0954  
9      12458                                             0.0958  
10     2136                                              0.0962  


In [10]:
joblib.dump(model_v2,                  '../data/model_v2/knn_model_v2.joblib')
save_npz('../data/model_v2/X_knn_norm_v2.npz', X_knn_norm_v2)
np.save('../data/model_v2/album_ids_annotated_v2.npy', album_ids_annotated_v2)
np.save('../data/model_v2/has_features_v2.npy',        has_features)

print('Saved:')
print('  ../data/model_v2/knn_model_v2.joblib')
print('  ../data/model_v2/X_knn_norm_v2.npz')
print('  ../data/model_v2/album_ids_annotated_v2.npy')
print('  ../data/model_v2/has_features_v2.npy')

Saved:
  ../data/model_v2/knn_model_v2.joblib
  ../data/model_v2/X_knn_norm_v2.npz
  ../data/model_v2/album_ids_annotated_v2.npy
  ../data/model_v2/has_features_v2.npy
